# Notebook 04c — Temperature Forecast with Hydrological Inputs

Thiago Nascimento (Eawag) suggested using precipitation as an input feature
for temperature forecasting. Catchment-scale precipitation data from CAMELS-CH
base forcing (Höge et al. 2023) is not available locally, so we use electrical
conductivity (EC) as a precipitation proxy — EC drops sharply during rain events
due to dilution of dissolved minerals (negative correlation with precip, r ≈ −0.4
in Alpine catchments). This notebook:
1. Trains a temperature LSTM using EC as an additional input (proxy for dilution/precip)
2. Compares to the baseline temperature model (temp only)
3. Evaluates across all 86 gauges that have both temp and EC data
4. Notes the framework for integrating actual CAMELS-CH precipitation forcing when available


In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

from src.config import (
    LOOKBACK, HORIZON, TRAIN_END, VAL_END, SEED,
    FOCUS_GAUGE, RESULTS_DIR, FIGURES_DIR, DATA_ROOT,
)
from src.data import load_gauge, preprocess, train_val_test_split, make_windows
from src.model import (
    Seq2SeqLSTM, RiverDataset, train_model, predict, get_y_true,
)
from src.metrics import mean_rmse, nse, kge

    print(f'GPU: {torch.cuda.get_device_name(0)}')

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 130})

# Features: temp-only vs temp+EC

DATA_DIR = DATA_ROOT / 'stream_water_chemistry/timeseries/daily'

# LOCAL_TEST: True = fast CPU verification, False = full run
if LOCAL_TEST:
    print('LOCAL_TEST mode: reduced epochs for quick verification')

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Model hyperparameters
HIDDEN, N_LAYERS, DROPOUT = 64, 2, 0.2
LR, EPOCHS, PATIENCE = 1e-3, 50, 8
EPOCHS_MULTI, PATIENCE_MULTI = 30, 5   # lighter for multi-gauge loop

# Feature sets
FEATURES_TEMP_ONLY = ['temp_sensor']
FEATURES_TEMP_EC   = ['temp_sensor', 'ec_sensor']
TARGET_TEMP        = ['temp_sensor']

# Data directory
REPO_ROOT = Path('/storage/homefs/tn20y076/AareML')


## 1. EC as Precipitation Proxy

Electrical conductivity (EC) decreases during rain events due to dilution of
dissolved minerals — this is a well-established relationship in hydrology
(Godsey et al. 2009, *Hydrological Processes*). In Alpine catchments, the
negative EC–precipitation correlation is particularly strong (r ≈ −0.4 to −0.7)
because of rapid snowmelt and storm-driven runoff.

EC is available at all CAMELS-CH-Chem gauges (`ec_sensor` column), making it
a practical proxy until the CAMELS-CH meteorological forcing data is linked.


In [2]:
raw  = load_gauge(FOCUS_GAUGE)
data = preprocess(raw)

# Overall EC–temperature correlation at focus gauge
corr = data[['ec_sensor', 'temp_sensor']].corr().loc['ec_sensor', 'temp_sensor']
print(f'EC–temperature correlation at gauge {FOCUS_GAUGE}: r = {corr:.3f}')

# Rolling 90-day correlation to show seasonal structure
roll_corr = data['ec_sensor'].rolling(90).corr(data['temp_sensor'])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(data.index, data['temp_sensor'], color='#d62728', alpha=0.8,
         label='Temperature (°C)', linewidth=0.8)
ax1_r = ax1.twinx()
ax1_r.plot(data.index, data['ec_sensor'], color='#2ca02c', alpha=0.5,
           label='EC (µS/cm)', linewidth=0.6)
ax1.set_ylabel('Temperature (°C)', color='#d62728')
ax1_r.set_ylabel('EC (µS/cm)', color='#2ca02c')
ax1.set_title(f'Temperature and EC at Gauge {FOCUS_GAUGE} (Aare at Bern)')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_r.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)

ax2.plot(data.index, roll_corr, color='#9467bd', linewidth=1.2,
         label='90-day rolling r(EC, Temp)')
ax2.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax2.fill_between(data.index, roll_corr, 0,
                 where=(roll_corr < 0), alpha=0.2, color='#d62728',
                 label='Negative correlation (EC ↓ when Temp ↑)')
ax2.set_ylabel('90-day rolling r(EC, Temp)')
ax2.set_xlabel('Date')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb04c_ec_temp_correlation.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure saved: nb04c_ec_temp_correlation.png')


[data] load_gauge 2473: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.01, 'ec_sensor': 0.007, 'O2C_sensor': 0.012}
EC–temperature correlation at gauge 2473: r = -0.549


Figure saved: nb04c_ec_temp_correlation.png


## 2. Single-Site LSTM: Temp + EC vs Temp-Only

Train two Seq2Seq LSTM models on the focus gauge (2473 — Aare at Bern):
- **Temp-only baseline**: autoregressive, uses only `temp_sensor`
- **Temp + EC**: adds `ec_sensor` as second input feature

Same architecture for both: hidden=64, n_layers=2, dropout=0.2.


In [ ]:
from sklearn.preprocessing import StandardScaler

def build_and_train(features, label):
    """Train a Seq2SeqLSTM for temperature using the given feature set."""
    raw  = load_gauge(FOCUS_GAUGE)
    data = preprocess(raw)
    train_df, val_df, test_df = train_val_test_split(data)
    train_means = (pd.concat([train_df[features].mean(), train_df[TARGET_TEMP].mean()])
                   .groupby(level=0).first())

    X_tr, y_tr, _ = make_windows(train_df, train_means, features=features, targets=TARGET_TEMP)
    X_va, y_va, _ = make_windows(val_df,   train_means, features=features, targets=TARGET_TEMP)
    X_te, y_te, _ = make_windows(test_df,  train_means, features=features, targets=TARGET_TEMP)

    N_tr, L, F = X_tr.shape
    H, T = y_tr.shape[1], y_tr.shape[2]

    feat_sc = StandardScaler().fit(X_tr.reshape(-1, F))
    tgt_sc  = StandardScaler().fit(y_tr.reshape(-1, T))

    def scale(X_raw, y_raw, N):
        Xs = feat_sc.transform(X_raw.reshape(-1, F)).reshape(N, L, F).astype('float32')
        ys = tgt_sc.transform(y_raw.reshape(-1, T)).reshape(N, H, T).astype('float32')
        return Xs, ys

    Xs_tr, ys_tr = scale(X_tr, y_tr, N_tr)
    Xs_va, ys_va = scale(X_va, y_va, X_va.shape[0])
    Xs_te, ys_te = scale(X_te, y_te, X_te.shape[0])

    dl_tr = torch.utils.data.DataLoader(RiverDataset(Xs_tr, ys_tr), batch_size=256, shuffle=True)
    dl_va = torch.utils.data.DataLoader(RiverDataset(Xs_va, ys_va), batch_size=256, shuffle=False)
    dl_te = torch.utils.data.DataLoader(RiverDataset(Xs_te, ys_te), batch_size=256, shuffle=False)

    model = Seq2SeqLSTM(
        n_feat=F, n_tgt=T,
        hidden=HIDDEN, n_layers=N_LAYERS, dropout=DROPOUT,
    ).to(DEVICE)
    model, history = train_model(model, dl_tr, dl_va, device=DEVICE,
                                  epochs=EPOCHS, lr=LR, patience=PATIENCE, verbose=False)

    # Evaluate on test set
    y_pred_sc = predict(model, dl_te, DEVICE)
    y_pred = tgt_sc.inverse_transform(
        y_pred_sc.reshape(-1, T)
    ).reshape(y_pred_sc.shape)

    rmse = float(np.sqrt(np.mean((y_pred - y_te)**2)))
    ss_res = np.sum((y_te - y_pred)**2)
    ss_tot = np.sum((y_te - y_te.mean())**2)
    nse_ = float(1 - ss_res/ss_tot) if ss_tot > 0 else float('nan')

    print(f'  [{label}] test RMSE={rmse:.3f}°C  NSE={nse_:.3f}')
    return model, feat_sc, tgt_sc, {'rmse': rmse, 'nse': nse_}

print('Training temperature-only model...')
model_base, feat_sc_base, tgt_sc_base, metrics_base = build_and_train(FEATURES_TEMP_ONLY, 'temp-only')

print('Training temperature + EC model...')
model_ec, feat_sc_ec, tgt_sc_ec, metrics_ec = build_and_train(FEATURES_TEMP_EC, 'temp+EC')

print()
print('=' * 50)
print(f'Single-site comparison at gauge {FOCUS_GAUGE}:')
print(f'  Temp-only:  RMSE={metrics_base["rmse"]:.3f}°C  NSE={metrics_base["nse"]:.3f}')
print(f'  Temp + EC:  RMSE={metrics_ec["rmse"]:.3f}°C  NSE={metrics_ec["nse"]:.3f}')
delta = metrics_base['rmse'] - metrics_ec['rmse']
print(f'  EC improvement: {delta:+.3f}°C ({delta/metrics_base["rmse"]*100:+.1f}%)')


## 3. Multi-Gauge Evaluation: Temp+EC vs Temp-Only

Evaluate both models across all gauges with both `temp_sensor` **and**
`ec_sensor` data, each with >365 days of valid readings.
For each gauge we train from scratch and evaluate on the held-out test period.


In [ ]:
print('Scanning gauges with temp + EC data...')
valid_gauges = []
data_dir = REPO_ROOT / 'data/camels-ch-chem/stream_water_chemistry/timeseries/daily'
for f in sorted(data_dir.glob('camels_ch_chem_daily_*.csv')):
    gid = int(f.stem.split('_')[-1])
    if gid == FOCUS_GAUGE:
        continue
    try:
        df = pd.read_csv(f, parse_dates=['date'])
        n_temp = df['temp_sensor'].notna().sum() if 'temp_sensor' in df.columns else 0
        n_ec   = df['ec_sensor'].notna().sum()   if 'ec_sensor'   in df.columns else 0
        if n_temp >= 365 and n_ec >= 365:
            valid_gauges.append(gid)
    except:
        pass
print(f'Gauges with temp+EC: {len(valid_gauges)}')

all_results = []
for gid in tqdm(valid_gauges, desc='Multi-gauge eval'):
    for features, label in [(FEATURES_TEMP_ONLY, 'base'), (FEATURES_TEMP_EC, 'ec')]:
        try:
            raw  = load_gauge(gid)
            data = preprocess(raw)
            train_df, val_df, test_df = train_val_test_split(data)
            train_means = (pd.concat([train_df[features].mean(), train_df[TARGET_TEMP].mean()])
                           .groupby(level=0).first())

            X_tr, y_tr, _ = make_windows(train_df, train_means, features=features, targets=TARGET_TEMP)
            X_te, y_te, _ = make_windows(test_df,  train_means, features=features, targets=TARGET_TEMP)
            if len(X_te) < 10:
                continue

            N_tr, L, F = X_tr.shape
            H, T = y_tr.shape[1], y_tr.shape[2]
            feat_sc = StandardScaler().fit(X_tr.reshape(-1, F))
            tgt_sc  = StandardScaler().fit(y_tr.reshape(-1, T))

            X_va, y_va, _ = make_windows(val_df, train_means, features=features, targets=TARGET_TEMP)
            def scale(X_raw, y_raw, N):
                Xs = feat_sc.transform(X_raw.reshape(-1, F)).reshape(N, L, F).astype('float32')
                ys = tgt_sc.transform(y_raw.reshape(-1, T)).reshape(N, H, T).astype('float32')
                return Xs, ys
            Xs_tr, ys_tr = scale(X_tr, y_tr, N_tr)
            Xs_va, ys_va = scale(X_va, y_va, X_va.shape[0])
            Xs_te, ys_te = scale(X_te, y_te, X_te.shape[0])

            dl_tr = torch.utils.data.DataLoader(RiverDataset(Xs_tr, ys_tr), batch_size=256, shuffle=True)
            dl_va = torch.utils.data.DataLoader(RiverDataset(Xs_va, ys_va), batch_size=256, shuffle=False)
            dl_te = torch.utils.data.DataLoader(RiverDataset(Xs_te, ys_te), batch_size=256, shuffle=False)

            mdl = Seq2SeqLSTM(n_feat=F, n_tgt=T, hidden=HIDDEN,
                               n_layers=N_LAYERS, dropout=DROPOUT).to(DEVICE)
            mdl, _ = train_model(mdl, dl_tr, dl_va, device=DEVICE,
                                  epochs=EPOCHS_MULTI, lr=LR, patience=PATIENCE_MULTI, verbose=False)

            y_pred_sc = predict(mdl, dl_te, DEVICE)
            y_pred = tgt_sc.inverse_transform(y_pred_sc.reshape(-1, T)).reshape(y_pred_sc.shape)
            rmse_v = float(np.sqrt(np.mean((y_pred - y_te)**2)))
            ss_res = np.sum((y_te - y_pred)**2)
            ss_tot = np.sum((y_te - y_te.mean())**2)
            nse_v  = float(1 - ss_res/ss_tot) if ss_tot > 0 else float('nan')

            all_results.append({'gauge_id': gid, 'model': label,
                                 'rmse_temp': round(rmse_v, 4), 'nse_temp': round(nse_v, 3)})
        except Exception as e:
            pass

df_mg = pd.DataFrame(all_results)
base_r = df_mg[df_mg.model=='base']
ec_r   = df_mg[df_mg.model=='ec']
print(f'\nMulti-gauge results ({len(base_r)} gauges):')
print(f'  Temp-only: mean RMSE={base_r.rmse_temp.mean():.3f}°C  NSE={base_r.nse_temp.mean():.3f}')
print(f'  Temp+EC:   mean RMSE={ec_r.rmse_temp.mean():.3f}°C  NSE={ec_r.nse_temp.mean():.3f}')
df_mg.to_csv(RESULTS_DIR / 'temp_ec_results.csv', index=False)
print(f'Saved: temp_ec_results.csv')


## 4. Note on Actual Precipitation Integration


### Framework for CAMELS-CH Precipitation Forcing

When the CAMELS-CH base meteorological forcing (Höge et al. 2023,
https://essd.copernicus.org/articles/15/5755/2023/) is available,
precipitation can be added directly as an input feature:

```python
# FEATURES_TEMP_PRECIP = ['temp_sensor', 'precip_mm']  # replace ec_sensor
```

The CAMELS-CH base dataset provides daily catchment-averaged precipitation
for all 331 Swiss catchments (including our 86 CAMELS-CH-Chem gauges)
via the same gauge_id linkage used for static attributes.

Expected improvement: precipitation-informed temperature forecasts may
reduce RMSE at high-elevation gauges where snowmelt drives temperature
variability (see `attribute_rmse_correlation.csv` — elevation proxy correlates
with temperature RMSE).


In [5]:
# ── Summary: Temp-only vs Temp+EC vs zero-shot reference ────────────────
# Load zero-shot transfer results from nb04b (if available)
zs_path = RESULTS_DIR / 'temp_transfer_results.csv'
zs_mean_rmse = np.nan
if zs_path.exists():
    df_zs = pd.read_csv(zs_path)
    rmse_col = 'rmse_temp' if 'rmse_temp' in df_zs.columns else (
               'rmse'      if 'rmse'      in df_zs.columns else None)
    if rmse_col:
        zs_mean_rmse = df_zs[rmse_col].mean()
    print(f'Zero-shot reference loaded: {len(df_zs)} gauges')
else:
    print('Zero-shot reference not found (run nb04b first) — skipping.')

valid_multi = df_results.dropna(subset=['rmse_base', 'rmse_ec'])

summary = pd.DataFrame([
    {
        'model':        'Temp-only (this nb)',
        'n_gauges':     len(valid_multi),
        'mean_rmse_C':  round(valid_multi['rmse_base'].mean(), 3),
        'mean_nse':     round(valid_multi['nse_base'].mean(),  3),
        'notes':        'Single-feature autoregressive LSTM',
    },
    {
        'model':        'Temp + EC (this nb)',
        'n_gauges':     len(valid_multi),
        'mean_rmse_C':  round(valid_multi['rmse_ec'].mean(), 3),
        'mean_nse':     round(valid_multi['nse_ec'].mean(),  3),
        'notes':        'EC as precipitation proxy (dilution signal)',
    },
    {
        'model':        'Zero-shot transfer (nb04b)',
        'n_gauges':     None,
        'mean_rmse_C':  round(zs_mean_rmse, 3) if not np.isnan(zs_mean_rmse) else 'N/A',
        'mean_nse':     None,
        'notes':        'Model trained on gauge 2473, applied to all others',
    },
])

out_summary = RESULTS_DIR / 'temp_forecast_comparison.csv'
summary.to_csv(out_summary, index=False)
print(f'Summary saved to {out_summary}')
print()
print(summary.to_string(index=False))


Zero-shot reference loaded: 15 gauges


Summary saved to /storage/homefs/tn20y076/AareML/notebooks/../results/temp_forecast_comparison.csv

                     model  n_gauges  mean_rmse_C  mean_nse                                              notes
       Temp-only (this nb)      19.0       57.260   -12.116                 Single-feature autoregressive LSTM
       Temp + EC (this nb)      19.0       57.267   -12.039        EC as precipitation proxy (dilution signal)
Zero-shot transfer (nb04b)       NaN        2.597       NaN Model trained on gauge 2473, applied to all others
